### Load and Filter the Dataset

#### 
We will use the glaiveai/glaive-function-calling-v2 dataset.    
Since the raw dataset contains chat interactions not leading to function calling,    
we first filter the data and later we'll balance the samples. 

In [1]:
import multiprocessing
from datasets import load_dataset

max_seq_length = 512
dataset_size = 'small'
train_eval_split = 0.1
train_test_split = 0.01
seed = 42
dataset_path = 'glaiveai/glaive-function-calling-v2'
fn_calling_dataset = load_dataset(
    dataset_path, split='train',
    num_proc=multiprocessing.cpu_count()
)
# Select samples that contain either a function call or a message indicating inability to call a function.
dataset_reduced = fn_calling_dataset.filter(
    lambda x: "I'm sorry" in x["chat"] or "functioncall" in x["chat"]
).shuffle(seed=seed)
dataset_reduced

c:\Users\quzhe\anaconda3\envs\fine_tuning_py312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['system', 'chat'],
    num_rows: 78392
})

We now "balance" the dataset to include both cases - when a function call is found and not -     
and then split it into training and testing sets. This will help us balance the dataset between    
the scenarios we want the model to improve:    
-Knowing when it can't do a function call;    
-How to perform a function call when it can.

In [2]:
from datasets import concatenate_datasets

def get_dataset_size(dataset_size):
    if dataset_size == "small":
        missed_amount = 200
        found_amount = 600
    elif dataset_size == "medium":
        missed_amount = 350
        found_amount = 750
    elif dataset_size == "large":
        missed_amount = 375
        found_amount = 825
    return missed_amount, found_amount

# Reserve a portion of the data for testing.
test_amount = max(int(train_test_split * dataset_reduced.num_rows), 25)
dataset_reduced_train = dataset_reduced.select(range(dataset_reduced.num_rows - test_amount))
# Determine the number of samples for each scenario.
missed_amount, found_amount = get_dataset_size(dataset_size)
dataset_train_missed = dataset_reduced_train.filter(
    lambda x: "I'm sorry" in x["chat"] and not ("functioncall" in x["chat"])
).select(range(missed_amount))
dataset_train_found = dataset_reduced_train.filter(
    lambda x: not ("I'm sorry" in x["chat"]) and "functioncall" in x["chat"]
).select(range(found_amount))
# Concatenate the two balanced datasets.
dataset_final_train = concatenate_datasets([dataset_train_missed, dataset_train_found])
# The reduced dataset now contains a small balanced mix of samples
dataset_final_train

Dataset({
    features: ['system', 'chat'],
    num_rows: 800
})

In [3]:
dataset_final_train

Dataset({
    features: ['system', 'chat'],
    num_rows: 800
})

In [14]:
dataset_final_train["chat"][10]

"USER: Can you please order a pizza for me?\n\n\nASSISTANT: I'm sorry, but I can't assist with that. My capabilities are currently limited to retrieving stock prices. I don't have the ability to place orders for food or other items. <|endoftext|>\n\n\n"

### Converting the Dataset Format

We'll be using the TRL library to handle a couple of steps, such as input tokenization,    
so we need to transform the dataset samples into a format the TRL trainer class expects.    
The dataset contains system and chat entries. For example, a sample from the dataset looks like:

>>> SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -    
{     
    "name": "convert_currency",     
    "description": "Convert the amount from one currency to another",    
    ...    
}    
-----    
USER: Hi, I need to convert 500 US dollars to Euros. Can you help me with that?    
ASSISTANT: {"name": "convert_currency", "arguments": '{"amount": 500, "from_currency": "USD", "to_currency": "EUR"}'}     
FUNCTION RESPONSE: {"converted_amount": 425.50, ...}    
ASSISTANT: Sure, 500 US dollars is approximately 425.50 Euros.    

We convert the original format (with system and chat fields) into a list messages:    

In [4]:
import re
from typing import List, Any, Dict, Tuple

def chat_str_to_messages(chat_str: str) -> Dict[str, Tuple[str, str]]:
    try:
        # Limit the chat to the point before the first function response.
        chat_until_function_call = chat_str[: next(re.finditer(r"FUNCTION\sRESPONSE\:", chat_str)).start()].strip()
    except StopIteration:
        chat_until_function_call = chat_str.strip()
    # use regex to find all user and assistant messages.
    matches = re.findall(
        r"(USER|ASSISTANT):\s(.*?)(?=\n\n|$)", chat_until_function_call, re.DOTALL
    )
    chat_interaction = [
        (matchh[0], matchh[1].replace(" <|endoftext|>", "").strip())
        for matchh in matches
    ]
    return chat_interaction
def transform_dataset_format(data_from_sample: List[Any]) -> Dict[str, List[Dict[str, str]]]:
    texts = []
    system_prompts = list(map(lambda x: re.split(r"SYSTEM\:\s", x)[1].strip(), data_from_sample["system"]))
    chats = list(map(chat_str_to_messages, data_from_sample["chat"]))
    for systemprompt, chatnow in zip(system_prompts, chats):
        messages = [{"role": "system", "content": systemprompt}] + [
            {"role": role.lower(), "content": msg} for role, msg in chatnow
        ]
        texts.append(messages)
    return {"messages": texts}
dataset_train = dataset_final_train.map(
    transform_dataset_format, batched=True,
    remove_columns=dataset_final_train.column_names,
)
dataset_train

Dataset({
    features: ['messages'],
    num_rows: 800
})

In [5]:
dataset_train[0]

{'messages': [{'content': 'You are a helpful assistant with access to the following functions. Use them if required -\n{\n    "name": "calculate_distance",\n    "description": "Calculate the distance between two locations",\n    "parameters": {\n        "type": "object",\n        "properties": {\n            "start_location": {\n                "type": "string",\n                "description": "The starting location"\n            },\n            "end_location": {\n                "type": "string",\n                "description": "The ending location"\n            }\n        },\n        "required": [\n            "start_location",\n            "end_location"\n        ]\n    }\n}',
   'role': 'system'},
  {'content': 'Can you please book a flight for me from New York to Los Angeles?',
   'role': 'user'},
  {'content': "I'm sorry, but as an AI, I don't have the capability to book flights. My current function allows me to calculate the distance between two locations. If you need to know th

Finally, we split the dataset between training and validation:

In [6]:
dataset_train_eval= dataset_train.train_test_split(test_size=train_eval_split)
dataset_train_eval

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 720
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 80
    })
})

### Loading the Pre-Trained DeepSeek Model and Configuring LoRA

Now that the dataset is ready, we load the DeepSeek model, set up the LoRA configuration,     
and let the GPU burn! Let’s check the amount of trainable parameters.    



In [7]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
model_size = '1.5B'
lora_r = 4 # this will vary the amount of trainable parameters - it's correlated with the performance gains.
lora_alpha = 16 # according to the paper, this is the best value for most tasks.
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"] # Modules to apply LoRA - it's correlated with the amount of trainable parameters.
# Load model and tokenizer.
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(
   model_name, padding=True, truncation=True, max_length=max_seq_length
)
# Set up the LoRA configuration.
lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=0.1,
    target_modules=target_modules,
    init_lora_weights="gaussian",
    task_type="CAUSAL_LM",
    inference_mode=False
)
# Wrap the model with LoRA and check the amount of trainable parameters
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 1,089,536 || all params: 1,778,177,536 || trainable%: 0.0613


### Configuring and Running the Fine-Tuning Process

#### Customizing Trainer Class

Using the TRL library, we set up a simple supervised fine-tuning trainer.     
I extended the original SFTTrainer class to ensure that the tokenizer properly handles padding to the specified max_seq_length.     
Not sure if there’s an issue when tokenizing with padding,     
but the SFTTrainer class seems to ignore the configuration to add max_seq_length padding to input.

In [8]:
import warnings
from datasets import Dataset
from typing import Optional, Callable
from trl import SFTTrainer

class CustomSFTTrainer(SFTTrainer):
    def _prepare_non_packed_dataloader(
        self,
        processing_class,
        dataset,
        dataset_text_field: str,
        max_seq_length,
        formatting_func: Optional[Callable] = None,
        add_special_tokens=True,
        remove_unused_columns=True,
    ):
        # Inspired from: https://huggingface.co/learn/nlp-course/chapter7/6?fw=pt
        
        
        # def tokenize(element):

        #     outputs = processing_class(
        #         element[dataset_text_field] if formatting_func is None else formatting_func(element),
        #         add_special_tokens=add_special_tokens,
        #         truncation=True,
        #         padding="max_length",
        #         max_length=max_seq_length,
        #         return_overflowing_tokens=False,
        #         return_length=False,
        #     )
        #     if formatting_func is not None and not isinstance(formatting_func(element), list):
        #         raise ValueError(
        #             "The `formatting_func` should return a list of processed strings since it can lead to silent bugs."
        #         )
        #     return {"input_ids": outputs["input_ids"], "attention_mask": outputs["attention_mask"]}
        
        
        def tokenize(element):
            # 取出字段内容
            value = element[dataset_text_field]
            # 如果是列表，并且列表内的每个元素是字典，且包含 "content" 键，则将所有消息拼接为字符串
            if isinstance(value, list) and all(isinstance(item, dict) and "content" in item for item in value):
                value = "\n".join(item["content"] for item in value)
            # 如果定义了 formatting_func，则使用 formatting_func 处理整个元素，否则直接使用转换后的 value
            # text_to_process = value if formatting_func is None else formatting_func(element)
            outputs = processing_class(
                value,
                add_special_tokens=add_special_tokens,
                truncation=True,
                padding="max_length",
                max_length=max_seq_length,
                return_overflowing_tokens=False,
                return_length=False,
            )
            if formatting_func is not None and not isinstance(formatting_func(element), list):
                raise ValueError(
                    "The `formatting_func` should return a list of processed strings since it can lead to silent bugs."
                )
            return {"input_ids": outputs["input_ids"], "attention_mask": outputs["attention_mask"]}


        signature_columns = ["input_ids", "labels", "attention_mask"]
        if dataset.column_names is not None:  # None for IterableDataset
            extra_columns = list(set(dataset.column_names) - set(signature_columns))
        else:
            extra_columns = []
        if not remove_unused_columns and len(extra_columns) > 0:
            warnings.warn(
                "You passed `remove_unused_columns=False` on a non-packed dataset. This might create some issues with "
                "the default collator and yield to errors. If you want to inspect dataset other columns (in this "
                f"case {extra_columns}), you can subclass `DataCollatorForLanguageModeling` in case you used the "
                "default collator and create your own data collator in order to inspect the unused dataset columns.",
                UserWarning,
            )
        map_kwargs = {
            "batched": True,
            "remove_columns": dataset.column_names if remove_unused_columns else None,
            "batch_size": self.dataset_batch_size,
        }
        if isinstance(dataset, Dataset):
            map_kwargs["num_proc"] = self.dataset_num_proc  # this arg is not available for IterableDataset
        tokenized_dataset = dataset.map(tokenize, **map_kwargs)
        return tokenized_dataset

#### Running the Trainer

There are a couple of parameters that have an impact on time/resources used during training,     
and it will depend on the type of hardware you are running.     
Newer GPUs would allow you to cast the model to torch.bfloat16,     
use tf32 dtype during training, etc, which would greatly improve training speed and GPU usage.     
This setup allows you to run with Google Colab’s free T4 GPU.     
You can also control the memory usage by configuring per_device_train_batch_size and gradient_accumulation_steps     
as the batch size is given by per_device_train_batch_size * gradient_accumulation_steps.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results

Reusing TensorBoard on port 6008 (pid 29188), started 0:39:45 ago. (Use '!kill 29188' to kill it.)

In [10]:
import pathlib
import os
from trl import SFTConfig

num_train_epochs = 1
max_steps = 200
bf16 = False
output_dir = 'results'
run_name = f"{model_name.split('/')[-1]}-fncall_peft-ds_{dataset_size}-lora_r_{lora_r}-use_qlora_False"
output_dir_final = os.path.join(output_dir, run_name)

# 调整 tokenizer 设置
tokenizer.padding_side = 'right'


def join_messages(example):
    # 拼接所有消息的 "content"
    if "messages" in example and isinstance(example["messages"], list):
        text = "\n".join([msg["content"] for msg in example["messages"] if "content" in msg])
        return {"text": text}
    return example

# 对训练集和测试集都进行处理
dataset_train_eval["train"] = dataset_train_eval["train"].map(join_messages)
dataset_train_eval["test"] = dataset_train_eval["test"].map(join_messages)



print("Creating trainer...")
pathlib.Path(output_dir_final).mkdir(parents=True, exist_ok=True)
training_args = SFTConfig(
    dataset_text_field="text",
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,  # 节省显存，可能会增加训练时间
    bf16=bf16,
    tf32=False,  # 对于 Ampere 及更新架构的 GPU，可使用 tf32 加速训练
    dataloader_pin_memory=False,  # 固定数据到内存
    torch_compile=False,  # 使用 PyTorch 的编译功能
    warmup_steps=50,
    max_steps=max_steps,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_strategy="steps",
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    eval_strategy="steps",
    logging_steps=10,
    output_dir=output_dir_final,
    optim="paged_adamw_8bit",
    remove_unused_columns=True,
    seed=seed,
    run_name=run_name,
    report_to="tensorboard",  # 使用 tensorboard 记录日志
    push_to_hub=False,
    eval_steps=25,
    # packing=False,  # 不使用 packing
)

trainer = CustomSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train_eval["train"],
    eval_dataset=dataset_train_eval["test"],
    processing_class=tokenizer,
    peft_config=lora_config
)

print("Training...")
trainer.train()


Map: 100%|██████████| 80/80 [00:00<00:00, 3371.39 examples/s]


Creating trainer...


Tokenizing eval dataset: 100%|██████████| 80/80 [00:00<00:00, 1944.90 examples/s]
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Training...


Step,Training Loss,Validation Loss
25,1.510600,1.038605
50,0.627500,0.597345
75,0.500000,0.486700
100,0.421000,0.435556
125,0.397900,0.383351
150,0.316000,0.336338
175,0.321600,0.319099
200,0.307900,0.316399


TrainOutput(global_step=200, training_loss=0.5760996985435486, metrics={'train_runtime': 1459.3968, 'train_samples_per_second': 2.193, 'train_steps_per_second': 0.137, 'total_flos': 7963680067737600.0, 'train_loss': 0.5760996985435486})

### Running Inference with the Fine-Tunned Model

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Helper function for chat generation.
def run_inout_pipe(chat_interaction, tokenizer, model):
    prompt = tokenizer.apply_chat_template(chat_interaction, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=512, pad_token_id=tokenizer.eos_token_id)
    outputs = outputs[:, inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# define the model and max_seq_length
max_seq_length = 512
model_name = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
# get latest checkpoint from the training sessions
checkpoint_lora_path = 'results/DeepSeek-R1-Distill-Qwen-1.5B-fncall_peft-ds_small-lora_r_4-use_qlora_False/checkpoint-200'

# Load base model and tokenizer.
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cuda")

tokenizer = AutoTokenizer.from_pretrained(
   model_name, padding=True, truncation=True, max_length=max_seq_length
)

offload_dir = "/temp/offload_dir" # In case the model needs to offload weights.
peft_model = PeftModel.from_pretrained(model, checkpoint_lora_path, offload_dir=offload_dir)

chat_interaction = [
    {
        "role": "system",
        "content": '''You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "convert_currency",
    "description": "Convert the amount from one currency to another",
    "parameters": {
        "type": "object",
        "properties": {
            "amount": {
                "type": "number",
                "description": "The amount to convert"
            },
            "from_currency": {
                "type": "string",
                "description": "The currency to convert from"
            },
            "to_currency": {
                "type": "string",
                "description": "The currency to convert to"
            }
        },
        "required": [
            "amount",
            "from_currency",
            "to_currency"
        ]
    }
}'''
},
    {
        "role": "user",
        "content": "Hi, I need to convert 500 US dollars to Euros. Can you help me with that?"
    }
]
print(run_inout_pipe(chat_interaction, tokenizer, peft_model))

Okay, so I need to convert 500 US dollars to Euros. Hmm, I'm not exactly sure how to do that. Let me think. I remember that currency conversion involves some exchange rate. But what's the exchange rate between the US dollar and the Euro? I think it's around 0.85 or something like that. Wait, no, I think it's 1 USD to 0.85 Euros. So that means one US dollar is equivalent to 0.85 Euros. 

So if I have 500 USD, I can multiply that by the exchange rate to get the amount in Euros. Let me do the math: 500 multiplied by 0.85. Hmm, 500 times 0.8 is 400, and 500 times 0.05 is 25, so adding those together gives me 425 Euros. 

Wait, but I'm not sure if that's the correct exchange rate. Maybe I should double-check. I recall that the current exchange rate from USD to Euros is approximately 0.85. So yes, 500 USD should convert to 425 Euros. 

Alternatively, I could use a conversion function to get the exact amount. But since I don't have that function, I'll go with the exchange rate I remember. So 